# Person 3: protected-mode smoke test (`wall_only`, `wall_gatekeeper`)

Runs the frozen `user_task_0` smoke test in the two protected modes, verifies the real Laya worker with
controlled actions, and builds the comparison table.

**Runtime:** Colab with an **A100** GPU (Runtime > Change runtime type). Laya runs on CPU; Qwen runs on the GPU.

**Two Python environments** (they need conflicting libraries):
- *Server env* (Colab's system Python): `vllm==0.11.0`, `transformers==4.57.1`, serves Qwen.
- *Runner env* (`/content/runner-env`): the repo, AgentDojo, the wall and Laya (`transformers==5.19.0`, `torch==2.14.1`). The experiment runner and the Laya worker run here and talk to vLLM over HTTP.

Everything is written to `/content/p3`. The last cell zips it; copy that zip to Drive before the runtime ends.

Run the cells in order. Do **not** edit the frozen configs. If a cell fails, stop and note the failure class in the report.

In [ ]:
# 1. Settings
REPO = 'https://github.com/AtharvaThorat/permission-that-learn-'
COMMIT = 'origin/person3-protected-smoke'  # has the pair_id fix + Person 3 scripts; use the merged main commit once merged.
# Person 2's baseline must come from the SAME commit, or the comparison table will say 'NO'.
WORK = '/content/permission-that-learn-'
OUT = '/content/p3'
RUNNER = '/content/runner-env'
MODEL_DIR = '/content/models/Qwen2.5-7B-Instruct'
LAYA_CACHE = '/content/laya-cache'          # must contain hub/models--convaiinnovations--laya/...
LAYA_REPO, LAYA_REV = 'convaiinnovations/laya', '7b928d828b7b0e022f929d9bd2e44165aa270148'
API_KEY = 'local-colab-person3'             # local loopback only; never a real credential
RUN_UNPROTECTED_HERE = False                # True only if Person 2's baseline is unavailable

import json, os, subprocess, sys, time, pathlib
os.makedirs(OUT, exist_ok=True)
PY = f'{RUNNER}/bin/python'

def sh(cmd, env=None, check=True, cwd=None):
    """Run a shell command, stream output, return combined text."""
    full = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or (WORK if os.path.isdir(WORK) else None), env=full,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {cmd}')
    return ''.join(lines)

OFFLINE = {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'BLACKBOX_ACTOR_API_KEY': API_KEY,
           'LAYA_CACHE_DIR': LAYA_CACHE}

In [ ]:
# 2. Clone the exact commit and record the machine
sh(f'git clone {REPO} {WORK}', cwd='/content', check=False)
sh(f'git fetch origin && git checkout --detach {COMMIT} && git rev-parse HEAD && git log -1 --oneline')
sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv; nproc; free -g; python3 --version')

## 3. Server env: vLLM (system Python)

In [ ]:
sh('pip install -q "vllm==0.11.0" "transformers==4.57.1"')
sh('python3 -c "import vllm, torch, transformers; print(vllm.__version__, torch.__version__, transformers.__version__)"')

## 4. Runner env: repo + AgentDojo + wall + Laya

In [ ]:
sh('pip install -q uv')
sh(f'uv venv {RUNNER} --python 3.12 --seed', check=False)
sh(f'uv pip install --python {PY} -r requirements-blackbox.txt -r requirements-blackbox-test.txt '
   f'hypothesis==6.170.5 -e ./wall -e .')
sh(f'uv pip check --python {PY}', check=False)
sh(f"{PY} -c \"import laya, torch, transformers, agentdojo, ptl.wall; "
   f"print('laya', laya.__version__ if hasattr(laya,'__version__') else 'ok', torch.__version__, transformers.__version__)\"")
sh(f'{PY} scripts/check_env.py', check=False)

In [ ]:
# 5. CPU regression suite in the runner env (expect 226 passed on person3-protected-smoke)
sh(f'{PY} -m pytest -q -p no:cacheprovider', check=False)

## 6. Download model assets (online), then everything else runs offline

In [ ]:
from huggingface_hub import snapshot_download
qwen_path = snapshot_download('Qwen/Qwen2.5-7B-Instruct', local_dir=MODEL_DIR)
laya_path = snapshot_download(LAYA_REPO, revision=LAYA_REV, cache_dir=f'{LAYA_CACHE}/hub')
print(qwen_path); print(laya_path)
# Record the Qwen commit actually downloaded (frozen config leaves model_revision null).
from huggingface_hub import HfApi
qwen_rev = HfApi().model_info('Qwen/Qwen2.5-7B-Instruct').sha
json.dump({'qwen_model_id': 'Qwen/Qwen2.5-7B-Instruct', 'qwen_revision': qwen_rev, 'qwen_path': MODEL_DIR,
           'laya_repo': LAYA_REPO, 'laya_revision': LAYA_REV, 'laya_snapshot': laya_path},
          open(f'{OUT}/model_assets.json', 'w'), indent=2)
assert pathlib.Path(laya_path).name == LAYA_REV, 'Laya snapshot revision mismatch'

## 7. Start Qwen on vLLM (loopback only)

In [ ]:
log = open(f'{OUT}/vllm.log', 'w')
server = subprocess.Popen(['bash', 'scripts/colab_vllm_start.sh', MODEL_DIR, '8000', 'Qwen2.5-7B-Instruct'],
                          cwd=WORK, env={**os.environ, 'BLACKBOX_ACTOR_API_KEY': API_KEY,
                                         'VLLM_GPU_MEMORY_UTILIZATION': '0.85'},
                          stdout=log, stderr=subprocess.STDOUT)
import urllib.request
for i in range(120):
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
        print(urllib.request.urlopen(req, timeout=5).read().decode()[:400]); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('vLLM exited; see /content/p3/vllm.log')
        time.sleep(10)
else:
    raise RuntimeError('vLLM not ready after 20 minutes')

In [ ]:
# 8. One validated tool-call request (Person 1 deliverable re-checked on this machine)
sh(f'{PY} scripts/validate_model_tool_call.py --base-url http://127.0.0.1:8000/v1 '
   f'--model Qwen2.5-7B-Instruct --output {OUT}/validation_response.json', env=OFFLINE)
print(open(f'{OUT}/validation_response.json').read())

## 9. Controlled cases

9a uses scripted gates (fixed expectations, same as the CPU run). 9b sends the same native actions through the
**real Laya worker**: this is the first test of `LayaBackend` inside the protected pipeline. Laya's decisions are
recorded, not asserted; the check is that nothing executes unless Laya returned ALLOW.

In [ ]:
sh(f'{PY} scripts/person3_controlled_cases.py --output {OUT}/controlled-scripted', env=OFFLINE, check=False)
sh(f'{PY} scripts/person3_controlled_cases.py --real-laya --laya-cache {LAYA_CACHE} '
   f'--output {OUT}/controlled-real-laya', env=OFFLINE, check=False)
for r in json.load(open(f'{OUT}/controlled-real-laya/controlled_summary.json')):
    print(r['case'], '| dispatched:', r['dispatched'], '| gate:', r['gate_decisions'])

## 10. Real runs: one fresh environment per mode, sequential

In [ ]:
def run_mode(config, label):
    sh(f'{PY} scripts/reset_workspace.py', env=OFFLINE)
    out = sh(f'{PY} -m experiment run --config {config} --output {OUT}/runs --allow-models', env=OFFLINE, check=False)
    start = out.rfind('{\n  "run_dir"')
    run_dir = json.loads(out[start:])['run_dir'] if start >= 0 else None
    print(f'\n>>> {label}: {run_dir}')
    return run_dir

RUNS = {}
if RUN_UNPROTECTED_HERE:
    RUNS['unprotected'] = run_mode('configs/experiments/frozen/unprotected.json', 'unprotected')
RUNS['wall_only'] = run_mode('configs/experiments/frozen/wall_only.json', 'wall_only')

In [ ]:
# 10b. wall_gatekeeper exactly as frozen. cache_dir is null in the frozen config, so this is expected to
# fail closed (ASK, MODEL_UNAVAILABLE, LOCAL_LAYA_SNAPSHOT_MISSING). Kept as evidence for the defect log.
RUNS['wall_gatekeeper_frozen_as_is'] = run_mode('configs/experiments/frozen/wall_gatekeeper.json', 'wall_gatekeeper (as frozen)')

In [ ]:
# 10c. wall_gatekeeper with only gatekeeper.cache_dir filled in (no other change).
cfg = json.load(open(f'{WORK}/configs/experiments/frozen/wall_gatekeeper.json'))
cfg['gatekeeper']['cache_dir'] = LAYA_CACHE
cfg['name'] = 'frozen-user-task-0-wall-gatekeeper-cache'
patched = f'{OUT}/wall_gatekeeper.cache_dir.json'
json.dump(cfg, open(patched, 'w'), indent=2)
sh(f'{PY} -m experiment plan --config {patched}', env=OFFLINE, check=False)
RUNS['wall_gatekeeper'] = run_mode(patched, 'wall_gatekeeper (cache_dir set)')
json.dump(RUNS, open(f'{OUT}/run_dirs.json', 'w'), indent=2)
RUNS

## 11. Comparison table

Put Person 2's baseline run folder in `BASELINE` (upload or copy it into `/content/p3/baseline/`).
The table reports comparability; Person 2 must have used the same commit.

In [ ]:
BASELINE = RUNS.get('unprotected') or ''   # e.g. '/content/p3/baseline/frozen-user-task-0-unpro-xxxxxxxx'
dirs = [d for d in [BASELINE, RUNS.get('wall_only'), RUNS.get('wall_gatekeeper'),
                    RUNS.get('wall_gatekeeper_frozen_as_is')] if d]
sh(f'{PY} scripts/person3_analyze_runs.py --run-dirs {" ".join(dirs)} --output {OUT}/report', check=False)
for d in dirs:
    sh(f'{PY} -m experiment aggregate --run-dir {d} > {d}/aggregate.json', check=False)
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/report/comparison.md').read()))

In [ ]:
# 12. Environment manifest + package everything
def out(cmd, py='python3'):
    try: return subprocess.check_output(cmd, shell=True, text=True, cwd=WORK).strip()
    except Exception as e: return f'ERROR {e}'
manifest = {
    'commit': out('git rev-parse HEAD'),
    'gpu': out('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader'),
    'server_env': out('python3 -c "import sys,vllm,torch,transformers;print(sys.version.split()[0],vllm.__version__,torch.__version__,transformers.__version__)"'),
    'runner_env': out(f"{PY} -c \"import sys,torch,transformers,importlib.metadata as m;print(sys.version.split()[0],torch.__version__,transformers.__version__,m.version('laya'),m.version('agentdojo'))\""),
    'runner_freeze': out(f'uv pip freeze --python {PY}').splitlines(),
    'assets': json.load(open(f'{OUT}/model_assets.json')),
    'runs': RUNS,
}
json.dump(manifest, open(f'{OUT}/environment-manifest.json', 'w'), indent=2)
server.terminate()
ZIP = f'/content/person3_smoke_{time.strftime("%Y%m%d_%H%M")}.zip'
sh(f'cd /content && zip -qr {ZIP} p3 && ls -la {ZIP}')
# Save it off the Colab machine: everything in /content is deleted when the session ends.
import shutil
from google.colab import drive, files
try:
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/person3_smoke', exist_ok=True)
    shutil.copy(ZIP, '/content/drive/MyDrive/person3_smoke/')
    print('Copied to Google Drive: MyDrive/person3_smoke/' + os.path.basename(ZIP))
except Exception as e:
    print('Drive copy skipped:', e)
files.download(ZIP)   # browser download to your Mac